# 🏰 Gatekeeper CTF Challenge — Level 2: The Gatekeeper Knows

## What Happened in Level 1?

In the previous challenge, you tampered with the training data and tricked the Gatekeeper into thinking the thief (ID = 10) was a Citizen.

But the attack did not go unnoticed.

## Scenario

The Gatekeeper is smarter than it looks. It somehow already knows exactly what a thief's record looks like — the features, the label, all of it. Nobody fully understands how, but it does.

So when someone changes **the thief's row** or **the Label column**, the Gatekeeper notices it immediately and rejects the submission.

The dataset is back to its original state. The Gatekeeper is correctly identifying the thief again.

## The New Constraint

The Gatekeeper is watching two specific things:

- **The last row** — that is the thief's record (ID = 10). You cannot change anything in it.
- **The last column** — that is the Label column. You cannot change any label values.

If either of these is modified in your submission, it will be automatically rejected.

---

## 🎯 Your Mission

Your goal remains the same — make the Gatekeeper misclassify the thief (ID = 10) as a **Citizen (Label = 0)**.

The dataset has two classes:
- **Label = 0 → Citizen** (innocent villager)
- **Label = 1 → Thief** (known criminal)

**Rules:**
- You may tamper with the data in **one of two ways**:
  - Use the **interactive table** provided in this notebook to edit values and save
  - Open `gatekeeper_dataset.csv` directly, make your changes, and save it as `tampered_data.csv`
- **Writing code to manipulate the data is not allowed**
- You **cannot** modify the last row (ID = 10) or the last column (Label)
- The model retrains on whatever is in `tampered_data.csv` at submission time
- A successful attack = Thief (ID 10) predicted as **Citizen (0)**
- 📤 **Submit only the `tampered_data.csv` file** — do NOT submit the notebook

---

## Pipeline Overview

```
Raw Data → Load CSV → Train Model → Predict Label for ID 10
                 ↑

          [You tamper here]```

## Step 1: Install Dependencies

Uninstall and reinstall all required packages to ensure you have the latest versions.

- `pandas`
- `scikit-learn`
- `ipywidgets`
- `ipydatagrid`
- `requests`

> Run this cell first before any other cell.

In [ ]:
%pip install --upgrade pandas scikit-learn ipywidgets ipydatagrid requests -q

In [ ]:
import requests

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_URL = ""
USERNAME = ""   # ← enter your CTF username
PASSWORD = ""   # ← enter your CTF password

# ── Login ─────────────────────────────────────────────────────────────────────
session = requests.Session()
resp = session.post(f"{BASE_URL}/api/notebook-login/", data={"username": USERNAME, "password": PASSWORD})
result = resp.json()

if result.get("success"):
    print(f"✅ Logged in as: {USERNAME}")
else:
    print(f"❌ Login failed: {result.get('message', 'Unknown error')}")

## Step 2: Load the Gatekeeper Dataset

This cell loads the dataset from `gatekeeper_dataset.csv`.

The dataset is a table where each row is one person. Here is what each column means:

- **ID** — a unique number for each person
- **Night_Activity** — how active the person is during the night
- **Trust_Index** — how much the community trusts this person
- **Contribution** — how much this person contributes positively to the village
- **Conflict_Score** — how many conflicts this person has been involved in
- **Label** — the true identity:
  - `0` = Citizen (innocent villager)
  - `1` = Thief (known criminal)

The model is trained on all rows and learns to predict whether someone is a Citizen or a Thief based on the four behavioral features.

> 🔒 **Do not modify the code cell below.** It loads the dataset and must run exactly as written.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("gatekeeper_dataset.csv")

## Step 3: Edit the Dataset

You have two options to tamper with the data:

**Option A — Use the interactive table below:**
1. Click any cell and type a new value
2. Make your changes to the eligible rows and columns (remember: last row and Label column are off-limits)
3. Click **Save as CSV** to save your tampered version to `tampered_data.csv`

**Option B — Edit the file directly:**
1. Open `gatekeeper_dataset.csv` in any spreadsheet app or text editor
2. Make your changes (do not touch the last row or the Label column)
3. Save the file as `tampered_data.csv` in this folder

> ⚠️ **Writing code to manipulate the data is not allowed.** Only the two methods above are permitted.

**Remember the constraints:**
- Do **not** change the last row (ID = 10 — the thief's record)
- Do **not** change the last column (Label)
- Everything else is fair game

**Buttons:**
| Button | Action |
|---|---|
| **Reset to Original** | Restores the table to the unmodified dataset |

| **Save as CSV** | Saves the current table to a CSV file |> 🔒 **Do not modify the code cell below.** It sets up the interactive table and the Save/Reset buttons — run it as-is.


## 📤 Submission

Once you are happy with your changes and confident the Gatekeeper will misclassify the thief:

1. Make sure your tampered data is saved as `tampered_data.csv` (via the table or your own code)
2. **Submit only the `tampered_data.csv` file** on the challenge platform

> ⚠️ Do NOT submit the notebook. Only the CSV file will be evaluated.

> ⚠️ The judge will automatically check that the last row (ID = 10) and the last column (Label) are unchanged. If either is modified, your submission will be rejected.

In [ ]:
import ipywidgets as widgets
from ipydatagrid import DataGrid
from IPython.display import display, clear_output, HTML

# Working copy of the dataset
edit_df = df.copy()

# --- Editable DataGrid ---
grid = DataGrid(
    edit_df,
    editable=True,
    layout={"height": "300px"},
    column_widths={
        "ID": 50,
        "Night_Activity": 120,
        "Trust_Index": 110,
        "Contribution": 110,
        "Conflict_Score": 120,
        "Label": 70,
    }
)

# --- Output area ---
output = widgets.Output()

# --- Buttons ---
reset_btn = widgets.Button(
    description="Reset to Original",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="180px", margin="10px 4px")
)

save_btn = widgets.Button(
    description="Save as CSV",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="160px", margin="10px 4px")
)

save_filename = widgets.Text(
    value="tampered_data.csv",
    description="Filename:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="280px", margin="10px 4px")
)

# --- Button callbacks ---
def on_reset(b):
    with output:
        clear_output()
        display(HTML("<p style='color:gray;font-family:Arial;'>Table reset to original data.</p>"))
    grid.data = df.copy()

def on_save(b):
    with output:
        clear_output()
        fname = save_filename.value.strip() or "tampered_data.csv"
        if not fname.endswith(".csv"):
            fname += ".csv"
        grid.data.to_csv(fname, index=False)
        display(HTML(f"<p style='color:green;font-family:Arial;font-weight:bold;'>✅ Saved to <code>{fname}</code></p>"))

reset_btn.on_click(on_reset)
save_btn.on_click(on_save)

# --- Layout ---
display(widgets.HTML("<h3 style='font-family:Arial'>✏️ Editable Gatekeeper Dataset — click any cell to edit</h3>"))
display(grid)
display(widgets.HBox([reset_btn, save_btn, save_filename]))
display(output)

## 📤 Submit Your Attack

Once you have made your changes and saved the file, run the cell below.

It will send your tampered dataset to the Gatekeeper evaluation server. The server will first verify that you respected the constraints (last row and Label column untouched), then retrain the model on your data and test whether the thief gets past the gate.

You will get an instant response telling you whether your attack worked, along with how confident the model was in its decision.

> 🔒 **Do not modify the code cell below.** It handles the submission to the evaluation server — run it as-is.

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────────
API_URL  = f"{BASE_URL}/evaluate/challenge-2"
CSV_FILE = "tampered_data.csv"

# ── Submit ────────────────────────────────────────────────────────────────────
if not os.path.exists(CSV_FILE):
    print(f"❌ '{CSV_FILE}' not found. Save your tampered data first using the table above.")
else:
    with open(CSV_FILE, "rb") as f:
        response = session.post(
            API_URL,
            files={"file": (CSV_FILE, f, "text/csv")},
        )

    result = response.json()

    print("=" * 50)
    if result.get("result") == "success":
        print("✅  CHALLENGE SOLVED!")
        print(f"🏴  Flag : {result['flag']}")
    else:
        print("❌  Not quite right. Keep trying!")

    print(f"\n💬  {result.get('message', '')}")
    print(f"\n📊  Model Confidence:")
    print(f"    Citizen : {result.get('citizen_confidence', 'N/A')}")
    print(f"    Thief   : {result.get('thief_confidence',   'N/A')}")
    print("=" * 50)